For this to work:

open "Teilnetz_Invest_Resultate.xlsx"
export the two sheets as csv, sheet1 should be called "data2.csv", sheet2 should be called "data3.csv"
press run :)

In [42]:
import pandas as pd
# 
# # Import the first dataset
df1 = pd.read_csv('data2.csv', sep=';', encoding='latin-1')

# Import the second dataset with the same settings
df2 = pd.read_csv('data3.csv', sep=';', encoding='latin-1')

# Append them together vertically
df = pd.concat([df1, df2], ignore_index=True)

C:\Users\z184922\AppData\Local\Temp\ipykernel_32460\773431156.py:4: DtypeWarning: Columns (0: Erneuerungskosten (Right)) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv('data2.csv', sep=';', encoding='latin-1')
C:\Users\z184922\AppData\Local\Temp\ipykernel_32460\773431156.py:7: DtypeWarning: Columns (0: Erneuerungskosten (Right)) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv('data3.csv', sep=';', encoding='latin-1')


In [43]:
df.head(5)


,Streckenkategorie Level 1,Streckenkategorie Level 2,Linie,Linie Level 2,Linie Level 3,Berichtsjahr,Gattungsname,Anlagentypname,Wiederbeschaffungswert (CHF),Substanz - Quality Aspect Grade,WBWxZK,Betrachtungsjahr,Threshold,Kostenstellen,Erneuerungskosten (Right),ER_Anlage_Jahr,Availability,Safety
0,Regionalstrecke,200250 - Le Day - (Vallorbe),200250,200 - Renens VD Ouest - Vallorbe,200250 - Le Day - (Vallorbe),2025,SAZ - Sicherungsanlagen und Zugbeeinflussung,Weichenheizungen,945000,1,0,2025,"3,8",200250,945000,56700,1,1
1,Regionalstrecke,200250 - Le Day - (Vallorbe),200250,200 - Renens VD Ouest - Vallorbe,200250 - Le Day - (Vallorbe),2025,SAZ - Sicherungsanlagen und Zugbeeinflussung,Weichenheizungen,945000,1,0,2025,"3,8",200250,945000,129360,1,1
2,Regionalstrecke,200250 - Le Day - (Vallorbe),200250,200 - Renens VD Ouest - Vallorbe,200250 - Le Day - (Vallorbe),2025,SAZ - Sicherungsanlagen und Zugbeeinflussung,Weichenheizungen,945000,1,0,2025,"3,8",200250,945000,"13631,8",1,1
3,Regionalstrecke,200300 - Vallorbe (communauté),200300,203 - Frasne - Vallorbe,200300 - Vallorbe (communauté),2025,NS - Niederspannungs- und Telekommunikationsan...,NSV_Kundeninformation,100000,5,0,2025,5,200300,0,2500,0,6
4,Hauptstrecke,230210 - Laufen - (Aesch),230210,230 - Delémont Est - Basel SBB Ost,230210 - Laufen - (Aesch),2025,NS - Niederspannungs- und Telekommunikationsan...,NSV_Kundeninformation,8000,1,0,2025,"4,7",230210,8000,200,0,1


In [44]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np


# Display basic info about the data
print("Data shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Available years:", sorted(df['Betrachtungsjahr'].unique()))
print("Sample Linie Level 3 values:", df['Linie Level 3'].unique()[:5])

Data shape: (1652001, 18)
Columns: ['Streckenkategorie Level 1', 'Streckenkategorie Level 2', 'Linie', 'Linie Level 2', 'Linie Level 3', 'Berichtsjahr', 'Gattungsname', 'Anlagentypname', 'Wiederbeschaffungswert (CHF)', 'Substanz - Quality Aspect Grade', 'WBWxZK', 'Betrachtungsjahr', 'Threshold', 'Kostenstellen', 'Erneuerungskosten (Right)', 'ER_Anlage_Jahr', 'Availability', 'Safety']
Available years: [np.int64(2025), np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), np.int64(2034), np.int64(2035), np.int64(2036), np.int64(2037), np.int64(2038), np.int64(2039), np.int64(2040), np.int64(2041), np.int64(2042), np.int64(2043), np.int64(2044), np.int64(2045), np.int64(2046), np.int64(2047), np.int64(2048), np.int64(2049), np.int64(2050)]
Sample Linie Level 3 values: <StringArray>
[  '200250 - Le Day - (Vallorbe)', '200300 - Vallorbe (communauté)',
      '230210 - Laufen - (Aesch)',   '226200 - Choindez - Delémont'

In [45]:
# Prepare data for visualization
# Group by Linie Level 3, Quality Grade Category, and Year, counting unique GUID

# Convert Quality Aspect Grade to numeric (handling any non-numeric values)
df['Quality_Grade_Numeric'] = pd.to_numeric(
    df['Substanz - Quality Aspect Grade'].astype(str).str.replace(',', '.'),
    errors='coerce'
)

# Create quality grade category (< 4 or > 4)
df['Quality_Category'] = df['Quality_Grade_Numeric'].apply(
    lambda x: 'Grade < 4' if pd.notna(x) and x < 4 else ('Grade > 4' if pd.notna(x) else 'No Grade')
)

# Get available years
years = sorted(df['Betrachtungsjahr'].dropna().unique())
print(f"Years available: {years}")

# Function to get data for a specific year based on general amount of assets
def get_year_data(year):
    df_year = df[df['Betrachtungsjahr'] == year].copy()
    
    # Count total assets (rows) per Linie Level 3 and Quality Category
    grouped = df_year.groupby(['Linie Level 3', 'Quality_Category']).size().reset_index(name='Count')
    
    return grouped

# Test with first year
years = sorted(df['Betrachtungsjahr'].dropna().unique())
test_data = get_year_data(years[0])
print(f"\nSample data for year {int(years[0])}:")
print(test_data.head(90))

Years available: [np.int64(2025), np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), np.int64(2034), np.int64(2035), np.int64(2036), np.int64(2037), np.int64(2038), np.int64(2039), np.int64(2040), np.int64(2041), np.int64(2042), np.int64(2043), np.int64(2044), np.int64(2045), np.int64(2046), np.int64(2047), np.int64(2048), np.int64(2049), np.int64(2050)]

Sample data for year 2025:
                                   Linie Level 3 Quality_Category  Count
0                 200100 - (Renens) - (Bussigny)        Grade < 4    367
1             200150 - Bussigny - Daillens [bif]        Grade < 4   2793
2             200150 - Bussigny - Daillens [bif]        Grade > 4   2430
3                 200200 - (Daillens) - (Le Day)        Grade < 4   1930
4                 200200 - (Daillens) - (Le Day)        Grade > 4   3314
5                   200250 - Le Day - (Vallorbe)        Grade < 4   1322
6                   200250 

In [46]:
import plotly.graph_objects as go

# 1. Get ALL unique Linie Level 3 names across all years combined
all_linies = set()
for year in years:
    df_year = get_year_data(year)
    all_linies.update(df_year['Linie Level 3'].dropna().unique())

# Sort them alphabetically
sorted_linies = sorted(list(all_linies))

# Prepare all frames for the slider
frames = []

for year in years:
    df_year = get_year_data(year)
    
    # Pivot data to have Grade < 4 and Grade > 4 as separate traces
    grade_lt4 = df_year[df_year['Quality_Category'] == 'Grade < 4'].set_index('Linie Level 3')['Count']
    grade_gt4 = df_year[df_year['Quality_Category'] == 'Grade > 4'].set_index('Linie Level 3')['Count']
    
    # Reindex with the global sorted lines (adds missing lines with 0 count)
    grade_lt4 = grade_lt4.reindex(sorted_linies, fill_value=0)
    grade_gt4 = grade_gt4.reindex(sorted_linies, fill_value=0)
    
    # Create frame for this year
    frame = go.Frame(
        data=[
            go.Bar(
                x=grade_lt4.values,
                y=grade_lt4.index,
                name='Quality < 4',
                orientation='h',
                marker=dict(color='#006E09'),
                hovertemplate='%{y}<br>Count: %{x}<extra></extra>'
            ),
            go.Bar(
                x=grade_gt4.values,
                y=grade_gt4.index,
                name='Quality > 4',
                orientation='h',
                marker=dict(color='#FF0000'),
                hovertemplate='%{y}<br>Count: %{x}<extra></extra>'
            )
        ],
        name=str(int(year))
    )
    frames.append(frame)


sorted_linies = sorted(df['Linie Level 3'].dropna().unique())

# Create initial figure using the first year's data, reindexed to the global list
initial_data = get_year_data(years[0])
grade_lt4_initial = initial_data[initial_data['Quality_Category'] == 'Grade < 4'].set_index('Linie Level 3')['Count']
grade_gt4_initial = initial_data[initial_data['Quality_Category'] == 'Grade > 4'].set_index('Linie Level 3')['Count']
grade_lt4_initial = grade_lt4_initial.reindex(sorted_linies, fill_value=0)
grade_gt4_initial = grade_gt4_initial.reindex(sorted_linies, fill_value=0)
fig = go.Figure(
    data=[
        go.Bar(
            x=grade_lt4_initial.values,
            y=grade_lt4_initial.index,
            name='Quality < 4',
            orientation='h',
            marker=dict(color="#006E09"),
            hovertemplate='%{y}<br>Count: %{x}<extra></extra>'
        ),
        go.Bar(
            x=grade_gt4_initial.values,
            y=grade_gt4_initial.index,
            name='Quality > 4',
            orientation='h',
            marker=dict(color="#FF0000"),
            hovertemplate='%{y}<br>Count: %{x}<extra></extra>'
        )
    ],
    frames=frames
)

# Add year slider
sliders = [
    {
        'active': 0,
        'yanchor': 'top',
        'y': 0,
        'xanchor': 'left',
        'x': 0.1,
        'currentvalue': {
            'prefix': 'Year: ',
            'visible': True,
            'xanchor': 'center',
            'font': {'size': 16}
        },
        'transition': {'duration': 300},
        'pad': {'b': 10, 't': 50},
        'len': 0.9,
        'steps': [
            {
                'args': [
                    [str(int(year))],
                    {
                        'frame': {'duration': 300, 'redraw': True},
                        'mode': 'immediate',
                        'transition': {'duration': 300}
                    }
                ],
                'method': 'animate',
                'label': str(int(year))
            }
            for year in years
        ]
    }
]

fig.update_layout(
    sliders=sliders,
    title='Unique Assets by Railway Line (Linie Level 3) - Quality Grade Comparison',
    xaxis_title='Count of Unique Assets ',
    yaxis_title='Linie Level 3',
    
    # ---> INSERT THIS YAXIS DICTIONARY HERE <---
    yaxis=dict(
        type='category',
        categoryorder='array',
        categoryarray=sorted_linies
    ),
    # -------------------------------------------
    
    barmode='stack',
    height=max(900, len(sorted_linies) * 20),
    hovermode='closest',
    showlegend=True,
    legend=dict(x=1.02, y=1, xanchor='left', yanchor='top')
)

fig.show()


In [81]:
#Reltative Anteile

import plotly.graph_objects as go

# 1. Get ALL unique Linie Level 3 names across all years combined
all_linies = set()
for year in years:
    df_year = get_year_data(year)
    all_linies.update(df_year['Linie Level 3'].dropna().unique())

# Sort them alphabetically
sorted_linies = sorted(list(all_linies))

# Prepare all frames for the slider
frames = []

for year in years:
    df_year = get_year_data(year)
    
    # Pivot data to have Grade < 4 and Grade > 4 as separate traces
    grade_lt4 = df_year[df_year['Quality_Category'] == 'Grade < 4'].set_index('Linie Level 3')['Count']
    grade_gt4 = df_year[df_year['Quality_Category'] == 'Grade > 4'].set_index('Linie Level 3')['Count']
    
    # Reindex with the global sorted lines (adds missing lines with 0 count)
    grade_lt4 = grade_lt4.reindex(sorted_linies, fill_value=0)
    grade_gt4 = grade_gt4.reindex(sorted_linies, fill_value=0)
    
    # --- CALCULATE RELATIVE PERCENTAGES ---
    total = grade_lt4 + grade_gt4
    # .fillna(0) handles division by zero if a line has a total count of 0 in a specific year
    relative_lt4 = (grade_lt4 / total * 100).fillna(0)
    relative_gt4 = (grade_gt4 / total * 100).fillna(0)
    # --------------------------------------
    
    # Create frame for this year
    frame = go.Frame(
        data=[
            go.Bar(
                x=relative_lt4.values,
                y=relative_lt4.index,
                customdata=grade_lt4.values,  # Pass absolute count to hover
                name='Good Quality',
                orientation='h',
                marker=dict(color='#006E09'),
                hovertemplate='%{y}<br>Percentage: %{x:.1f}%<br>Count: %{customdata}<extra></extra>'
            ),
            go.Bar(
                x=relative_gt4.values,
                y=relative_gt4.index,
                customdata=grade_gt4.values,  # Pass absolute count to hover
                name='Critical Quality',
                orientation='h',
                marker=dict(color='#FF0000'),
                hovertemplate='%{y}<br>Percentage: %{x:.1f}%<br>Count: %{customdata}<extra></extra>'
            )
        ],
        name=str(int(year))
    )
    frames.append(frame)


# Create initial figure using the first year's data, reindexed to the global list
initial_data = get_year_data(years[0])
grade_lt4_initial = initial_data[initial_data['Quality_Category'] == 'Grade < 4'].set_index('Linie Level 3')['Count']
grade_gt4_initial = initial_data[initial_data['Quality_Category'] == 'Grade > 4'].set_index('Linie Level 3')['Count']
grade_lt4_initial = grade_lt4_initial.reindex(sorted_linies, fill_value=0)
grade_gt4_initial = grade_gt4_initial.reindex(sorted_linies, fill_value=0)

# --- CALCULATE INITIAL RELATIVE PERCENTAGES ---
total_initial = grade_lt4_initial + grade_gt4_initial
relative_lt4_initial = (grade_lt4_initial / total_initial * 100).fillna(0)
relative_gt4_initial = (grade_gt4_initial / total_initial * 100).fillna(0)
# ----------------------------------------------

fig = go.Figure(
    data=[
        go.Bar(
            x=relative_lt4_initial.values,
            y=relative_lt4_initial.index,
            customdata=grade_lt4_initial.values,
            name='Good Quality',
            orientation='h',
            marker=dict(color="#006E09"),
            hovertemplate='%{y}<br>Percentage: %{x:.1f}%<br>Count: %{customdata}<extra></extra>'
        ),
        go.Bar(
            x=relative_gt4_initial.values,
            y=relative_gt4_initial.index,
            customdata=grade_gt4_initial.values,
            name='Critical Quality',
            orientation='h',
            marker=dict(color="#FF0000"),
            hovertemplate='%{y}<br>Percentage: %{x:.1f}%<br>Count: %{customdata}<extra></extra>'
        )
    ],
    frames=frames
)

# Add year slider
sliders = [
    {
        'active': 0,
        'yanchor': 'top',
        'y': 0,
        'xanchor': 'left',
        'x': 0.0,
        'currentvalue': {
            'prefix': 'Year: ',
            'visible': True,
            'xanchor': 'center',
            'font': {'size': 16}
        },
        'transition': {'duration': 300},
        'pad': {'b': 2, 't': 50},
        'len': 1.3,
        'steps': [
            {
                'args': [
                    [str(int(year))],
                    {
                        'frame': {'duration': 300, 'redraw': True},
                        'mode': 'immediate',
                        'transition': {'duration': 300}
                    }
                ],
                'method': 'animate',
                'label': str(int(year))
            }
            for year in years
        ]
    }
]


fig.update_layout(
    sliders=sliders,
    autosize=True,
    # --- FIXED TITLE SEGMENT ---
    title=dict(
        text='Quality of Unique Assets',
        x=0.05,             # Placed at the far-left coordinate edge
        y=0.98,            
        xanchor='left',    # Left edge of the title text anchors to x=0.0
        yanchor='top',
        font=dict(
            size=24,       
            weight='bold'  
        )
    ),
    # ---------------------------
    xaxis=dict(
        title='Distribution [%]',
        range=[0, 100],          
        autorange=False,         
        automargin=True,          
    ),
    yaxis_title='',
    yaxis=dict(
        type='category',
        categoryorder='array',
        categoryarray=sorted_linies,
        automargin=True,          
    ),
    barmode='stack',
    height=max(1000, len(sorted_linies) * 20),
    hovermode='closest',
    showlegend=True,
    
    legend=dict(
        x=0.5,             
        y=1.08,            
        xanchor='center',  
        yanchor='top',
        orientation='h',   
        font=dict(
            size=20  
        ),
        itemsizing='constant' 
    ),
    
    margin=dict(
        l=10,  
        r=10,  
        t=120, 
        b=20  
    ) 
)


fig.write_html("relative_quality_grade_comparison.html", include_plotlyjs='cdn', full_html=True)
fig.show()